# Practice - 00 - Workspace, storage, and synthetic source data

Prerequisites / execution context: Databricks Python notebook; UC compute; existing writable catalog. Change CATALOG if workspace is not your catalog. Creates three dedicated schemas, a volume and lab tables. Re-running resets the main Bronze table.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R05: Free Edition limitations](https://docs.databricks.com/aws/en/getting-started/free-edition-limitations); [R08: Unity Catalog volumes](https://docs.databricks.com/aws/en/volumes/); [R58: UC privileges reference](https://docs.databricks.com/aws/en/data-governance/unity-catalog/access-control/privileges-reference); [R71: Import/export notebooks](https://docs.databricks.com/aws/en/notebooks/notebook-export-import).


**Try first:** complete each YOUR CODE / YOUR ANSWER cell. Only then run the solution-display cell and click Reveal. Solutions are feedback text, never automatically executed. Cells marked check compare your own implementation. Work top to bottom; do not use Run All while placeholders remain.

**Isolation:** the default prefix in this edition is dea_riccardo_practice. Run the practice setup before later practice notebooks. For opt-in cells, complete the code before enabling the flag.


In [ ]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "workspace"
PREFIX = "dea_riccardo_practice"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


In [ ]:
# Run once. This helper displays a solution; it never executes solution code.
import base64 as _practice_base64

def show_solution(encoded_html):
    panel = _practice_base64.b64decode(encoded_html).decode("utf-8")
    try:
        renderer = displayHTML  # Databricks
    except NameError:
        from IPython.display import display, HTML
        display(HTML(panel))
    else:
        renderer(panel)


## Coding task 1: Create the lab namespace

Using BRONZE, SILVER, GOLD, create each schema if absent. Create the managed volume BRONZE.landing. Execute one SQL statement per spark.sql call. Do not create a catalog.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE


# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAxIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDEuIENyZWF0ZSBvYmplY3RzIG9uZSBzdGF0ZW1lbnQgYXQgYSB0aW1lClByZWRpY3Qgd2hpY2ggcGVybWlzc2lvbiBlYWNoIHN0YXRlbWVudCBuZWVkcy4gVGhlIG5vdGVib29rIHVzZXMgYW4gZXhpc3RpbmcgY2F0YWxvZzsgaXQgZG9lcyBub3QgcmVxdWlyZSBDUkVBVEUgQ0FUQUxPRy4KCgpUaGUgcmVmZXJlbmNlIGltcGxlbWVudGF0aW9uIGlzIHNob3duIG5leHQuIENvbXBhcmUgeW91ciBkZXNpZ24gYW5kIHJlc3VsdHMgYmVmb3JlIGNvcHlpbmcgaXQuPC9kaXY+PC9kZXRhaWxzPg==')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAxIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPmZvciBzY2hlbWEgaW4gKEJST05aRSwgU0lMVkVSLCBHT0xEKToKICAgIHNwYXJrLnNxbChmJnF1b3Q7Q1JFQVRFIFNDSEVNQSBJRiBOT1QgRVhJU1RTIHtzY2hlbWF9JnF1b3Q7KQpzcGFyay5zcWwoZiZxdW90O0NSRUFURSBWT0xVTUUgSUYgTk9UIEVYSVNUUyB7QlJPTlpFfS5sYW5kaW5nJnF1b3Q7KQpwcmludChCUk9OWkUsIFNJTFZFUiwgR09MRCwgVk9MVU1FKQo8L2NvZGU+PC9wcmU+PC9kZXRhaWxzPg==')


## Provided fixture - run unchanged
This supplies the synthetic input, not an exercise solution. Run after creating schemas and volume. Do not edit amounts, IDs or timestamps: later checks depend on this fixture. Re-running resets Bronze, so finish/stop dependent streams before resetting.


In [ ]:
import json
rows = [
 ("e1","o1","00035","100.00","2026-11-01 10:00:00",1,["web","priority"]),
 ("e1","o1","00035","100.00","2026-11-01 10:00:00",1,["web","priority"]),
 ("e2","o2","00037","50.00","2026-11-01 11:00:00",1,["store"]),
 ("e3","o1","00035","120.00","2026-11-02 10:00:00",2,["web"]),
 ("e4","o3",None,"20.00","2026-11-02 12:00:00",1,[]),
 ("e5","o4","00038","-5.00","2026-11-02 13:00:00",1,["web"]),
 ("e6","o5","00038","oops","2026-11-02 14:00:00",1,["mobile"]),
 ("e7","o6","00999","30.00","2026-11-02 15:00:00",1,["mobile"])
]
cols = ["event_id","order_id","customer_id","amount","event_time","seq","tags"]
payload = [dict(zip(cols,r)) for r in rows]
dbutils.fs.put(f"{VOLUME}/seed/orders.json", "\n".join(json.dumps(r) for r in payload), True)
raw_schema = "event_id STRING, order_id STRING, customer_id STRING, amount STRING, event_time STRING, seq LONG, tags ARRAY<STRING>"
raw = spark.read.schema(raw_schema).json(f"{VOLUME}/seed")
raw.write.format("delta").mode("overwrite").saveAsTable(f"{BRONZE}.orders_raw")
customers = spark.createDataFrame([("00035","IT"),("00037","FR"),("00038","IT")], "customer_id STRING, country STRING")
customers.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER}.customers")
assert raw.count()==8
assert customers.count()==3
raw.show(truncate=False)


## Coding task 2: Inspect the Bronze table

Display the detailed Delta metadata and operation history for BRONZE.orders_raw. Explain which result describes storage and which describes changes.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE


# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAyIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIFZlcmlmeSBhbmQgZXhwbGFpbgpPcGVuIENhdGFsb2cgRXhwbG9yZXIuIExvY2F0ZSB0aGUgdGhyZWUgc2NoZW1hcywgbWFuYWdlZCBEZWx0YSB0YWJsZSBhbmQgdm9sdW1lLiBFeHBsYWluIHdoeSB0aGUgSlNPTiBmaWxlIGJlbG9uZ3MgaW4gYSB2b2x1bWUgYW5kIHRoZSBxdWVyeWFibGUgcmVjb3JkcyBiZWxvbmcgaW4gYSB0YWJsZS4gSW5zcGVjdCB0YWJsZSBkZXRhaWxzIGFuZCBoaXN0b3J5LgoKClRoZSByZWZlcmVuY2UgaW1wbGVtZW50YXRpb24gaXMgc2hvd24gbmV4dC4gQ29tcGFyZSB5b3VyIGRlc2lnbiBhbmQgcmVzdWx0cyBiZWZvcmUgY29weWluZyBpdC48L2Rpdj48L2RldGFpbHM+')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAyIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPnNwYXJrLnNxbChmJnF1b3Q7REVTQ1JJQkUgREVUQUlMIHtCUk9OWkV9Lm9yZGVyc19yYXcmcXVvdDspLnNob3codHJ1bmNhdGU9RmFsc2UpCnNwYXJrLnNxbChmJnF1b3Q7REVTQ1JJQkUgSElTVE9SWSB7QlJPTlpFfS5vcmRlcnNfcmF3JnF1b3Q7KS5zaG93KHRydW5jYXRlPUZhbHNlKQo8L2NvZGU+PC9wcmU+PC9kZXRhaWxzPg==')


## Written / design task 1

Count unique event IDs and order IDs in the supplied fixture. Explain why they differ from raw row count. When is rerunning the fixture safe?


**YOUR ANSWER**

_Write your prediction, design or explanation here before opening the solution._


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgd3JpdHRlbiB0YXNrIDEgZmVlZGJhY2s8L3N1bW1hcnk+PGRpdiBzdHlsZT0id2hpdGUtc3BhY2U6cHJlLXdyYXA7b3ZlcmZsb3ctd3JhcDphbnl3aGVyZSI+IyMgRXhlcmNpc2UKV2l0aG91dCBjaGFuZ2luZyB0aGUgc291cmNlLCBzdGF0ZSBob3cgbWFueSB1bmlxdWUgZXZlbnQgSURzIGFuZCBob3cgbWFueSB1bmlxdWUgb3JkZXIgSURzIGV4aXN0LiBTb2x1dGlvbjogc2V2ZW4gZXZlbnQgSURzOyBzaXggb3JkZXIgSURzLiBSZS1ydW4gdGhlIHNldHVwIG9ubHkgd2hlbiB5b3UgaW50ZW5kIHRvIHJlc2V0IHRoZSBtYWluIGRhdGFzZXQuIEluZ2VzdGlvbiBub3RlYm9va3MgdXNlIHNlcGFyYXRlIGxvY2F0aW9ucy4KPC9kaXY+PC9kZXRhaWxzPg==')
